In [1]:
# Cell 1 - setup
from pathlib import Path
import os, json, time, requests

OUTDIR = Path("D:/projectsnuclear/papers")
OUTDIR.mkdir(parents=True, exist_ok=True)

METADATA_FILE = Path("arxiv-metadata-oai-snapshot.json")
if not METADATA_FILE.exists():
    raise FileNotFoundError("Download 'arxiv-metadata-oai-snapshot.json' from Kaggle and place it here.")
print("Ready. OUTDIR:", OUTDIR)

Ready. OUTDIR: D:\projectsnuclear\papers


In [2]:
# Cell 2 - extract nucl-th and nucl-ex IDs
NUCLEAR_IDS = []
with METADATA_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        paper = json.loads(line)
        cats = paper.get("categories","").split()
        if "nucl-th" in cats or "nucl-ex" in cats:
            NUCLEAR_IDS.append(paper["id"])

print("Found nuclear IDs:", len(NUCLEAR_IDS))
Path("nuclear_ids.txt").write_text("\n".join(NUCLEAR_IDS))

Found nuclear IDs: 76739


926316

In [3]:
# Cell 3 - resume setup
downloaded_file = Path("downloaded_ids.txt")
if downloaded_file.exists():
    downloaded_ids = set(downloaded_file.read_text().splitlines())
else:
    downloaded_ids = set()
print(f"Already downloaded: {len(downloaded_ids)} / {len(NUCLEAR_IDS)}")

Already downloaded: 42561 / 76739


In [4]:
# Cell 4 - download loop (robust, resume-capable)
from pathlib import Path
import os, time, requests

headers = {"User-Agent": "local-nuclear-crawler (contact: your-email@example.com)"}
# Replace the contact email above with a real address — polite and helpful if arXiv needs to reach you.

def download_one(arxiv_id, outdir: Path, headers, max_retries=1, timeout=10):
    url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
    outpath = outdir / f"{arxiv_id}.pdf"
    part = outdir / f"{arxiv_id}.pdf.part"
    if outpath.exists():
        return True
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, headers=headers, stream=True, timeout=timeout)
            if r.status_code == 200:
                with open(part, "wb") as fh:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            fh.write(chunk)
                # atomic replace (works on Windows)
                os.replace(part, outpath)
                return True
            else:
                print(f"{arxiv_id}: HTTP {r.status_code}")
        except Exception as e:
            print(f"{arxiv_id}: error {e} (attempt {attempt})")
        # exponential backoff between retries
        time.sleep(5 * attempt)
    # if here, failed — clean partial
    if part.exists():
        try:
            part.unlink()
        except:
            pass
    return False

# main loop (safe single-threaded)
for idx, aid in enumerate(NUCLEAR_IDS, start=1):
    if aid in downloaded_ids:
        continue
    print(f"[{idx}/{len(NUCLEAR_IDS)}] -> {aid}")
    ok = download_one(aid, OUTDIR, headers)
    if ok:
        with open(downloaded_file, "a", encoding="utf-8") as f:
            f.write(aid + "\n")
        downloaded_ids.add(aid)
    else:
        print(f"Failed to download {aid}. It will be retried on next run.")
    # polite delay to avoid being throttled or blocked

[1818/76739] -> 0802.0828
0802.0828: HTTP 403
Failed to download 0802.0828. It will be retried on next run.
[1889/76739] -> 0802.3002
0802.3002: HTTP 403
Failed to download 0802.3002. It will be retried on next run.
[2166/76739] -> 0804.3196
0804.3196: HTTP 403
Failed to download 0804.3196. It will be retried on next run.
[2719/76739] -> 0807.1464
0807.1464: HTTP 403
Failed to download 0807.1464. It will be retried on next run.
[3079/76739] -> 0809.2310
0809.2310: HTTP 403
Failed to download 0809.2310. It will be retried on next run.
[4228/76739] -> 0903.3129
0903.3129: HTTP 403
Failed to download 0903.3129. It will be retried on next run.
[4890/76739] -> 0907.1963
0907.1963: HTTP 403
Failed to download 0907.1963. It will be retried on next run.
[4923/76739] -> 0907.2799
0907.2799: HTTP 403
Failed to download 0907.2799. It will be retried on next run.
[5271/76739] -> 0909.0243
0909.0243: HTTP 403
Failed to download 0909.0243. It will be retried on next run.
[5754/76739] -> 0911.2763
09

KeyboardInterrupt: 